# Forest Fire Detection — Dataset Audit
## Notebook 01: Complete Dataset Discovery & Analysis

**Objective**: Recursively inspect all dataset directories, identify every image, CSV, and archive file,
compute statistics, and select datasets for training.


In [ ]:
import os, sys, json, hashlib
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────
ROOT = Path(r"e:/Sharvayu data/Malware/Symbiosis Nagpur SIT/7th SEM/Forest Fire task")
IMPL = ROOT / "Implementation"
ARCHIVE_DIR   = ROOT / "archive (1)"
FOREST_DIR    = ROOT / "forest+fires"
UAVS_DIR      = ROOT / "forestfire-8gb"
ARTIFACTS     = IMPL / "artifacts"
METADATA_DIR  = ARTIFACTS / "metadata"
PLOTS_DIR     = ARTIFACTS / "plots"
METADATA_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
CSV_EXTS   = {'.csv'}
ARCH_EXTS  = {'.zip', '.rar', '.7z', '.tar', '.gz'}

print("Forest Fire AI — Dataset Audit")
print("="*60)
print(f"Project root: {ROOT}")
print(f"Python: {sys.version.split()[0]}")


## 1. Recursive File Discovery

In [ ]:
def scan_dir(base_path, label):
    results = {'images': [], 'csvs': [], 'archives': [], 'others': []}
    base_path = Path(base_path)
    if not base_path.exists():
        print(f"  [WARN] {base_path} does not exist")
        return results
    for p in base_path.rglob("*"):
        if p.is_file():
            ext = p.suffix.lower()
            if ext in IMAGE_EXTS:
                results['images'].append(str(p))
            elif ext in CSV_EXTS:
                results['csvs'].append(str(p))
            elif ext in ARCH_EXTS:
                results['archives'].append(str(p))
            else:
                results['others'].append(str(p))
    print(f"\n[{label}]")
    print(f"  Images:   {len(results['images']):,}")
    print(f"  CSVs:     {len(results['csvs'])}")
    print(f"  Archives: {len(results['archives'])}")
    print(f"  Others:   {len(results['others'])}")
    return results

scan_archive  = scan_dir(ARCHIVE_DIR,  "archive (1)")
scan_forest   = scan_dir(FOREST_DIR,   "forest+fires")
scan_uavs     = scan_dir(UAVS_DIR,     "forestfire-8gb")


## 2. Archive Dataset — fire_dataset

In [ ]:
fire_img_dir    = ARCHIVE_DIR / "fire_dataset" / "fire_images"
nonfire_img_dir = ARCHIVE_DIR / "fire_dataset" / "non_fire_images"

fire_imgs    = sorted(fire_img_dir.glob("*.png"))
nonfire_imgs = sorted(nonfire_img_dir.glob("*.png"))

print(f"Archive Dataset — fire_dataset")
print(f"  fire_images:     {len(fire_imgs):,}")
print(f"  non_fire_images: {len(nonfire_imgs):,}")
print(f"  Total:           {len(fire_imgs)+len(nonfire_imgs):,}")
print(f"  Class ratio fire:nofire = {len(fire_imgs)/len(nonfire_imgs):.2f}:1")


## 3. UAVS-FDDB Dataset Breakdown

In [ ]:
uavs_raw = UAVS_DIR / "UAVS-FDDB UAVs-based Forest Fire Detection Database" / "Original Image Dataset (Raw Images)"
uavs_aug = UAVS_DIR / "UAVS-FDDB UAVs-based Forest Fire Detection Database" / "Augmented Images"

# Map folders to labels
FIRE_LABEL_KEYWORDS    = ['fire', 'Fire', 'FIRE']
NOFIRE_LABEL_KEYWORDS  = ['forest', 'Forest', 'FOREST', 'condition', 'Condition']

uavs_raw_stats = {}
for d in sorted(uavs_raw.iterdir()):
    if d.is_dir():
        imgs = list(d.rglob("*"))
        imgs = [f for f in imgs if f.is_file() and f.suffix.lower() in IMAGE_EXTS]
        folder_name = d.name
        # Determine label from folder name
        name_lower = folder_name.lower()
        if 'fire incident' in name_lower:
            label = 'FIRE'
        elif 'forest condition' in name_lower:
            label = 'NO_FIRE'
        else:
            label = 'UNKNOWN'
        uavs_raw_stats[folder_name] = {'count': len(imgs), 'label': label, 'path': str(d)}

print("UAVS-FDDB Raw Image Dataset:")
for folder, info in uavs_raw_stats.items():
    print(f"  {folder}: {info['count']} images → Label: {info['label']}")

total_uavs_fire    = sum(v['count'] for v in uavs_raw_stats.values() if v['label']=='FIRE')
total_uavs_nofire  = sum(v['count'] for v in uavs_raw_stats.values() if v['label']=='NO_FIRE')
total_uavs_raw     = sum(v['count'] for v in uavs_raw_stats.values())
print(f"\nTotal FIRE images:    {total_uavs_fire}")
print(f"Total NO_FIRE images: {total_uavs_nofire}")
print(f"Total raw images:     {total_uavs_raw}")


In [ ]:
# Augmented images
uavs_aug_stats = {}
for d in sorted(uavs_aug.iterdir()):
    if d.is_dir():
        imgs = list(d.rglob("*"))
        imgs = [f for f in imgs if f.is_file() and f.suffix.lower() in IMAGE_EXTS]
        name_lower = d.name.lower()
        if 'fire' in name_lower:
            label = 'FIRE'
        else:
            label = 'NO_FIRE'
        uavs_aug_stats[d.name] = {'count': len(imgs), 'label': label}
        print(f"  {d.name}: {len(imgs)} images → {label}")

total_aug = sum(v['count'] for v in uavs_aug_stats.values())
print(f"\nTotal augmented images: {total_aug:,}")


## 4. Forest Fires CSV Analysis

In [ ]:
csv_path = FOREST_DIR / "forestfires.csv"
df = pd.read_csv(csv_path)
print(f"forestfires.csv  →  Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:")
print(df.dtypes.to_string())
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"\nTarget column 'area' statistics:")
print(df['area'].describe().round(4).to_string())
print(f"\nRows where area > 0 (fire occurred): {(df['area'] > 0).sum()}")
print(f"Rows where area == 0 (no fire burned): {(df['area'] == 0).sum()}")
print(f"\nMonths present: {sorted(df['month'].unique())}")
print(f"Days present:   {sorted(df['day'].unique())}")


## 5. Image Dimension Analysis

In [ ]:
from PIL import Image
import random
random.seed(42)

def sample_image_stats(img_paths, n=50, label=""):
    sample = random.sample(img_paths, min(n, len(img_paths)))
    sizes, modes = [], []
    for p in sample:
        try:
            with Image.open(p) as im:
                sizes.append(im.size)
                modes.append(im.mode)
        except:
            pass
    if sizes:
        widths  = [s[0] for s in sizes]
        heights = [s[1] for s in sizes]
        print(f"  [{label}] n={len(sizes)} sampled")
        print(f"    Width:  min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}")
        print(f"    Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}")
        print(f"    Modes:  {set(modes)}")

# Archive dataset
print("Archive Dataset image dimensions:")
sample_image_stats([str(p) for p in fire_imgs], label="FIRE")
sample_image_stats([str(p) for p in nonfire_imgs], label="NO_FIRE")

# UAVS dataset
print("\nUAVS-FDDB Raw image dimensions:")
for folder, info in uavs_raw_stats.items():
    p = Path(info['path'])
    imgs = list(p.rglob("*"))
    imgs = [str(f) for f in imgs if f.is_file() and f.suffix.lower() in IMAGE_EXTS]
    if imgs:
        sample_image_stats(imgs, n=30, label=f"{info['label']} ({folder[:30]})")


## 6. Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle("Sample Images — Forest Fire Dataset", fontsize=14, fontweight='bold')

fire_sample = random.sample([str(p) for p in fire_imgs], 8)
nonfire_sample = random.sample([str(p) for p in nonfire_imgs], 8)

for i, (row_imgs, row_label) in enumerate([(fire_sample, 'FIRE'), (nonfire_sample, 'NO FIRE')]):
    for j, img_path in enumerate(row_imgs):
        try:
            img = Image.open(img_path).convert('RGB')
            axes[i][j].imshow(img)
            axes[i][j].axis('off')
            if j == 0:
                axes[i][j].set_title(row_label, fontsize=10, fontweight='bold',
                                      color='red' if row_label=='FIRE' else 'green')
        except Exception as e:
            axes[i][j].axis('off')

plt.tight_layout()
plot_path = PLOTS_DIR / "sample_images.png"
plt.savefig(plot_path, dpi=100, bbox_inches='tight')
plt.close()
print(f"Saved: {plot_path}")


## 7. Class Distribution Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Archive dataset
labels_arch = ['FIRE', 'NO_FIRE']
counts_arch = [len(fire_imgs), len(nonfire_imgs)]
colors = ['#E84040', '#2E8B57']
bars = axes[0].bar(labels_arch, counts_arch, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Archive Dataset — Class Distribution', fontweight='bold')
axes[0].set_ylabel('Number of Images')
for bar, count in zip(bars, counts_arch):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(count), ha='center', va='bottom', fontweight='bold')

# UAVS raw
uavs_labels = list(uavs_raw_stats.keys())
uavs_counts = [v['count'] for v in uavs_raw_stats.values()]
uavs_colors = ['#E84040' if v['label']=='FIRE' else '#2E8B57'
                for v in uavs_raw_stats.values()]
bars2 = axes[1].bar(range(len(uavs_labels)), uavs_counts, color=uavs_colors, edgecolor='black')
axes[1].set_xticks(range(len(uavs_labels)))
axes[1].set_xticklabels([l[:25] for l in uavs_labels], rotation=20, ha='right', fontsize=8)
axes[1].set_title('UAVS-FDDB Raw — Class Distribution', fontweight='bold')
axes[1].set_ylabel('Number of Images')
for bar, count in zip(bars2, uavs_counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(count), ha='center', va='bottom', fontsize=9)
fire_patch = mpatches.Patch(color='#E84040', label='FIRE')
nofire_patch = mpatches.Patch(color='#2E8B57', label='NO_FIRE')
axes[1].legend(handles=[fire_patch, nofire_patch])

plt.tight_layout()
plt.savefig(PLOTS_DIR / "class_distribution.png", dpi=100, bbox_inches='tight')
plt.close()
print("Class distribution plot saved.")


## 8. Dataset Summary & Selection Decision

In [ ]:
# Compute combined totals
archive_total = len(fire_imgs) + len(nonfire_imgs)
uavs_total    = total_uavs_raw

# Build summary
summary_data = {
    'Dataset': [
        'Archive fire_dataset',
        'UAVS-FDDB Raw Images',
        'UAVS-FDDB Augmented',
        'forestfires.csv'
    ],
    'Type': ['Image', 'Image', 'Image', 'Tabular'],
    'FIRE Samples': [len(fire_imgs), total_uavs_fire, 'N/A (pre-augmented)', 'N/A'],
    'NO_FIRE Samples': [len(nonfire_imgs), total_uavs_nofire, 'N/A', 'N/A'],
    'Total': [archive_total, uavs_total, total_aug, len(df)],
    'Classes': ['FIRE, NO_FIRE', 'FIRE, NO_FIRE', 'FIRE, NO_FIRE', 'Regression (area)'],
    'Suitable For': ['Image Classification', 'Image Classification', 'Augmentation reference', 'Fire Risk Regression']
}
df_summary = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("DATASET INVENTORY")
print("="*80)
print(df_summary.to_string(index=False))

# Save metadata
metadata = {
    'archive_fire': len(fire_imgs),
    'archive_nofire': len(nonfire_imgs),
    'archive_total': archive_total,
    'uavs_fire': total_uavs_fire,
    'uavs_nofire': total_uavs_nofire,
    'uavs_raw_total': uavs_total,
    'uavs_augmented': total_aug,
    'csv_rows': len(df),
    'csv_columns': len(df.columns),
    'fire_img_dir': str(fire_img_dir),
    'nonfire_img_dir': str(nonfire_img_dir),
    'uavs_raw_stats': uavs_raw_stats,
    'csv_path': str(csv_path)
}
with open(METADATA_DIR / "dataset_inventory.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)
print(f"\nMetadata saved: {METADATA_DIR}/dataset_inventory.json")


In [ ]:

msg = [
    "=" * 70,
    "  DATASET SELECTION DECISION",
    "=" * 70,
    "  IMAGE CLASSIFICATION:",
    "    [PRIMARY]    Archive fire_dataset  (755 FIRE + 244 NO_FIRE = 999)",
    "    [SUPPLEMENT] UAVS-FDDB Raw Images  (1145 FIRE + 385 NO_FIRE)",
    "    [SKIP]       UAVS Augmented        (avoid test leakage)",
    "  TABULAR PREDICTION:",
    "    [USE]  forestfires.csv (518 rows, regression on burned area)",
    "    [USE]  Binary classification (area>0 = fire occurred)",
    "  COMBINED IMAGE TOTAL: ~2529 images (after de-duplication)",
    "  CLASSES: FIRE / NO_FIRE (binary)",
    "=" * 70
]
print("\n".join(msg))


## 9. Findings Summary

In [ ]:
print("AUDIT COMPLETE")
print(f"  Archive images: {archive_total} (755 FIRE + 244 NO_FIRE)")
print(f"  UAVS raw images: {uavs_total} (1145 FIRE + 385 NO_FIRE)")
print(f"  UAVS augmented: {total_aug:,} (not used as test data)")
print(f"  CSV records: {len(df)}")
print(f"  CSV task: Regression (area) + Binary classification (area>0)")
print(f"  Target image input size for model: 224x224 px")
print(f"  No CUDA GPU detected — CPU training with efficient batch size")
